## Chain Using LangGraph

In this section we will see how we can build a simple chain using LangGraph that uses 4 important concepts.

- How to use chat messages as out graph state
- How to use chat models in graph nodes
- How to bind tools to out LLM in chat models
- How to execute the tools call in our graph nodes

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") # type: ignore
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") # type: ignore

### How to use chat messages as our graph state

### Messages

We can use messages which can be used to caputre different roles within a conversation. LangChain has various message types including HumanMessage, AIMessage, SystemMessage and ToolMessage. These represent a message from the user, from chat model, for the chat model to instruct behabior, and from a tool call.

Every message have these important components.

- content - content of the message
- name - Specify the name of author
- response_metadata - optionally, a dict of metadata(e.g., often populated by model provider fo AIMessages)

In [5]:
from langchain_core.messages import AIMessage, HumanMessage
from pprint import pprint

messages = [AIMessage(content=f"Please tell me how can i help", name="LLMModel")]
messages.append(HumanMessage(content=f"I want to learn coding", name = "Luis"))
messages.append(AIMessage(content=f"Which programming language want to learn?", name="LLMModel"))
messages.append(HumanMessage(content=f"I want to learn python programming language", name = "Luis"))

for m in messages:
    m.pretty_print()

================================== Ai Message ==================================
Name: LLMModel

Please tell me how can i help
================================ Human Message =================================
Name: Luis

I want to learn coding
================================== Ai Message ==================================
Name: LLMModel

Which programming language want to learn?
================================ Human Message =================================
Name: Luis

I want to learn python programming language


### Chat Models

We can use the sequence of message as input with the chatmodels using LLM's and OPENAI

In [7]:
from langchain_groq import ChatGroq
llm_groq = ChatGroq(model="qwen/qwen3-32b")
result = llm_groq.invoke(messages)
result

AIMessage(content='<think>\nOkay, the user wants to learn Python. Let me start by breaking down how to approach this. First, I need to figure out their current level. Since they haven\'t mentioned any experience, I should assume they\'re a beginner.\n\nI should outline a learning path. Start with the basics: variables, data types, control structures. Then move on to functions, data structures like lists and dictionaries. Maybe next step is working with files and error handling. After that, more advanced topics like OOP and libraries. Then introduce projects to apply what they\'ve learned.\n\nWait, the user might need resources. I should mention free platforms like Codecademy, freeCodeCamp, and Coursera. Also, books like "Automate the Boring Stuff" are great for beginners. Oh, and interactive tutorials on Replit or Jupyter Notebooks could be helpful.\n\nProjects are important for practice. Suggesting simple projects like a calculator, to-do list, or web scraper. Maybe mention contributi

In [8]:
result.response_metadata

{'token_usage': {'completion_tokens': 1528,
  'prompt_tokens': 54,
  'total_tokens': 1582,
  'completion_time': 3.1174018119999998,
  'completion_tokens_details': None,
  'prompt_time': 0.00192336,
  'prompt_tokens_details': None,
  'queue_time': 0.17368089,
  'total_time': 3.119325172},
 'model_name': 'qwen/qwen3-32b',
 'system_fingerprint': 'fp_2bfcc54d36',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

### Tools

Tools can be integrated with the LLM models to interact with external systems. External systems can be API's, third party tools.

Whenever a query is asked the model can choose to call the tool and this query is based on the natural language input and this will return an output that matches the tool's schema.